### Soal 1 — RLock pada Class

Buat sebuah class `RekeningBank` dengan method `setor(jumlah)` dan `tarik(jumlah)`, serta method `transfer(rekening_tujuan, jumlah)` yang **memanggil** `tarik()` pada dirinya sendiri dan `setor()` pada rekening tujuan. Gunakan `RLock` pada setiap instance. Jelaskan mengapa `RLock` diperlukan di sini dibanding `Lock` biasa.

#### Jawaban

In [7]:
import threading
import random
import subprocess
import sys
import os
import tempfile

# ---------- 1. Class RekeningBank dengan RLock ----------
class RekeningBank:
    def __init__(self, nomor, saldo_awal=0):
        self.nomor = nomor
        self.saldo = saldo_awal
        self._lock = threading.RLock()   # satu RLock per instance

    def setor(self, jumlah):
        with self._lock:
            self.saldo += jumlah

    def tarik(self, jumlah):
        with self._lock:
            if jumlah > self.saldo:
                raise ValueError(f"Saldo rekening {self.nomor} tidak cukup")
            self.saldo -= jumlah

    def transfer(self, rekening_tujuan, jumlah):
        # Kunci kedua rekening dengan urutan konsisten (berdasarkan nomor rekening)
        # supaya transfer A->B dan B->A yang berjalan bersamaan tidak deadlock.
        pertama, kedua = sorted([self, rekening_tujuan], key=lambda r: r.nomor)
        with pertama._lock:
            with kedua._lock:
                self.tarik(jumlah)              # mengunci self LAGI (reentrant)
                rekening_tujuan.setor(jumlah)   # mengunci tujuan LAGI (reentrant)

# ---------- 2. Uji sederhana ----------
a = RekeningBank("001", 1000)
b = RekeningBank("002", 500)
a.transfer(b, 300)
print(f"Setelah transfer 300 dari A ke B -> saldo A = {a.saldo}, saldo B = {b.saldo}")

try:
    a.transfer(b, 99999)
except ValueError as e:
    print("Transfer gagal (seperti yang diharapkan):", e)
print(f"Saldo setelah transfer gagal      -> saldo A = {a.saldo}, saldo B = {b.saldo} (tidak berubah)")

# ---------- 3. Uji konkuren: transfer bolak-balik dari banyak thread ----------
rekening = [RekeningBank(f"{i:03d}", 1000) for i in range(1, 4)]
total_awal = sum(r.saldo for r in rekening)

def pekerja(n_transfer):
    for _ in range(n_transfer):
        asal, tujuan = random.sample(rekening, 2)
        try:
            asal.transfer(tujuan, random.randint(1, 50))
        except ValueError:
            pass   # saldo tidak cukup -> lewati

threads = [threading.Thread(target=pekerja, args=(500,)) for _ in range(6)]
for t in threads: t.start()
for t in threads: t.join(timeout=10)

print("\nSemua thread selesai (tanpa deadlock):", not any(t.is_alive() for t in threads))
print("Saldo akhir :", [r.saldo for r in rekening])
print(f"Total awal = {total_awal}, total akhir = {sum(r.saldo for r in rekening)} -> konsisten: {total_awal == sum(r.saldo for r in rekening)}")

# ---------- 4. Bukti: versi dengan Lock biasa -> self-deadlock ----------
skrip_lock_biasa = '''
import threading

class RekeningBank:
    def __init__(self, saldo):
        self.saldo = saldo
        self._lock = threading.Lock()      # <-- Lock biasa, BUKAN RLock

    def setor(self, jumlah):
        with self._lock:
            self.saldo += jumlah

    def tarik(self, jumlah):
        with self._lock:
            self.saldo -= jumlah

    def transfer(self, tujuan, jumlah):
        with self._lock:                   # lock self diambil di sini ...
            print("transfer: sudah pegang lock, memanggil tarik()...", flush=True)
            self.tarik(jumlah)             # ... lalu diminta LAGI di sini -> macet
            tujuan.setor(jumlah)

a, b = RekeningBank(1000), RekeningBank(500)
t = threading.Thread(target=a.transfer, args=(b, 100))
t.start(); t.join()
print("selesai tanpa deadlock")
'''
path = os.path.join(tempfile.gettempdir(), "rekening_lock_biasa.py")
with open(path, "w") as f:
    f.write(skrip_lock_biasa)

try:
    hasil = subprocess.run([sys.executable, "-u", path], capture_output=True, text=True, timeout=3)
    print("\nOutput:", hasil.stdout)
except subprocess.TimeoutExpired as e:
    keluaran = e.stdout.decode() if isinstance(e.stdout, bytes) else (e.stdout or "(kosong)")
    print("\nTIMEOUT 3 detik -> SELF-DEADLOCK pada versi Lock biasa!")
    print("Output sebelum macet:", keluaran.strip())

Setelah transfer 300 dari A ke B -> saldo A = 700, saldo B = 800
Transfer gagal (seperti yang diharapkan): Saldo rekening 001 tidak cukup
Saldo setelah transfer gagal      -> saldo A = 700, saldo B = 800 (tidak berubah)

Semua thread selesai (tanpa deadlock): True
Saldo akhir : [1582, 392, 1026]
Total awal = 3000, total akhir = 3000 -> konsisten: True

TIMEOUT 3 detik -> SELF-DEADLOCK pada versi Lock biasa!
Output sebelum macet: transfer: sudah pegang lock, memanggil tarik()...


### Soal 2 — Deadlock dengan 3 Lock

Buat skenario deadlock yang melibatkan **tiga** lock dan **tiga** thread, dengan pola circular wait: Thread1 menunggu Lock-B (dipegang Thread2), Thread2 menunggu Lock-C (dipegang Thread3), Thread3 menunggu Lock-A (dipegang Thread1). Buktikan terjadinya deadlock menggunakan teknik `subprocess` + `timeout`, lalu perbaiki dengan urutan lock yang konsisten.

#### Jawaban

**Skenario deadlock:**

| Thread | Memegang | Meminta | Menunggu siapa |
|--------|----------|---------|----------------|
| Thread1 | Lock-A | Lock-B | Thread2 |
| Thread2 | Lock-B | Lock-C | Thread3 |
| Thread3 | Lock-C | Lock-A | Thread1 |

Rantai tunggu membentuk lingkaran tertutup: **Thread1 → Thread2 → Thread3 → Thread1**.



In [8]:
import os
import sys
import tempfile
import subprocess

# ---------- Skrip 1: DEADLOCK dengan 3 lock & 3 thread (circular wait A->B->C->A) ----------
skrip_deadlock_3 = '''
import threading, time, sys

def cetak(teks):
    sys.stdout.write(teks + "\\n"); sys.stdout.flush()   # 1x tulis -> baris tidak tercampur

lock_a = threading.Lock()
lock_b = threading.Lock()
lock_c = threading.Lock()

def thread1():                      # pegang A, minta B
    with lock_a:
        cetak("Thread1 pegang Lock-A, mau Lock-B...")
        time.sleep(0.3)
        with lock_b:
            cetak("Thread1 dapat A dan B")

def thread2():                      # pegang B, minta C
    with lock_b:
        cetak("Thread2 pegang Lock-B, mau Lock-C...")
        time.sleep(0.3)
        with lock_c:
            cetak("Thread2 dapat B dan C")

def thread3():                      # pegang C, minta A
    with lock_c:
        cetak("Thread3 pegang Lock-C, mau Lock-A...")
        time.sleep(0.3)
        with lock_a:
            cetak("Thread3 dapat C dan A")

ts = [threading.Thread(target=f) for f in (thread1, thread2, thread3)]
for t in ts: t.start()
for t in ts: t.join()
cetak("SELESAI - tidak ada deadlock")
'''

# ---------- Skrip 2: PERBAIKAN - semua thread mengambil lock dengan urutan A -> B -> C ----------
skrip_perbaikan_3 = '''
import threading, time, sys

def cetak(teks):
    sys.stdout.write(teks + "\\n"); sys.stdout.flush()   # 1x tulis -> baris tidak tercampur

lock_a = threading.Lock()
lock_b = threading.Lock()
lock_c = threading.Lock()

def thread1():                      # A lalu B (sudah sesuai urutan)
    with lock_a:
        cetak("Thread1 pegang Lock-A, mau Lock-B...")
        time.sleep(0.3)
        with lock_b:
            cetak("Thread1 dapat A dan B")

def thread2():                      # B lalu C (sudah sesuai urutan)
    with lock_b:
        cetak("Thread2 pegang Lock-B, mau Lock-C...")
        time.sleep(0.3)
        with lock_c:
            cetak("Thread2 dapat B dan C")

def thread3():                      # PERBAIKAN: sebelumnya C lalu A, sekarang A lalu C
    with lock_a:
        cetak("Thread3 pegang Lock-A, mau Lock-C...")
        time.sleep(0.3)
        with lock_c:
            cetak("Thread3 dapat A dan C")

ts = [threading.Thread(target=f) for f in (thread1, thread2, thread3)]
for t in ts: t.start()
for t in ts: t.join()
cetak("SELESAI - tidak ada deadlock")
'''

def jalankan(skrip, nama_file, batas_waktu):
    path = os.path.join(tempfile.gettempdir(), nama_file)
    with open(path, "w") as f:
        f.write(skrip)
    try:
        hasil = subprocess.run([sys.executable, "-u", path],
                               capture_output=True, text=True, timeout=batas_waktu)
        print(hasil.stdout)
        print(f"Kode keluar: {hasil.returncode} (0 = berhasil, tanpa timeout)")
    except subprocess.TimeoutExpired as e:
        keluaran = e.stdout.decode() if isinstance(e.stdout, bytes) else (e.stdout or "(kosong)")
        print(f"TIMEOUT setelah {batas_waktu} detik -- DEADLOCK TERKONFIRMASI!")
        print("Output sebelum macet total:")
        print(keluaran)

print("=== BAGIAN A: skenario deadlock 3 lock / 3 thread ===")
jalankan(skrip_deadlock_3, "deadlock_3_lock.py", batas_waktu=3)

print("\n=== BAGIAN B: setelah diperbaiki (urutan lock konsisten A -> B -> C) ===")
jalankan(skrip_perbaikan_3, "deadlock_3_lock_fixed.py", batas_waktu=5)

=== BAGIAN A: skenario deadlock 3 lock / 3 thread ===
TIMEOUT setelah 3 detik -- DEADLOCK TERKONFIRMASI!
Output sebelum macet total:
Thread1 pegang Lock-A, mau Lock-B...
Thread2 pegang Lock-B, mau Lock-C...
Thread3 pegang Lock-C, mau Lock-A...


=== BAGIAN B: setelah diperbaiki (urutan lock konsisten A -> B -> C) ===
Thread1 pegang Lock-A, mau Lock-B...
Thread2 pegang Lock-B, mau Lock-C...
Thread2 dapat B dan C
Thread1 dapat A dan B
Thread3 pegang Lock-A, mau Lock-C...
Thread3 dapat A dan C
SELESAI - tidak ada deadlock

Kode keluar: 0 (0 = berhasil, tanpa timeout)


### Soal 3 — Bounded Buffer dengan Kapasitas Berbeda

Ulangi eksperimen bagian 7 dengan `UKURAN_BUFFER` = 1, 3, 10, dan **2 producer + 2 consumer** sekaligus (bukan 1 producer). Amati dan jelaskan bagaimana ukuran buffer mempengaruhi pola interleaving antara producer dan consumer pada output.

#### Jawaban

**Pengaturan eksperimen:** desain tetap sama dengan bagian 7 (`slot_kosong` + `slot_terisi` + `mutex`), tetapi dibungkus fungsi `eksperimen(ukuran_buffer)` dengan 2 producer (masing-masing 8 item) dan 2 consumer (masing-masing mengambil 8 item). Producer dibuat **lebih cepat** (0,01–0,03 dtk) daripada consumer (0,03–0,06 dtk), supaya efek kapasitas buffer terlihat jelas. Agar interleaving mudah dibaca, setiap operasi pada buffer dicatat ke dalam `log` (di dalam `mutex`) lalu dicetak sebagai string `P` (producer menaruh) dan `C` (consumer mengambil). Fungsi ini juga menghitung berapa kali producer/consumer terpaksa **menunggu** karena buffer penuh/kosong.

**Hasil pengamatan (pola yang konsisten pada beberapa kali percobaan):**

| `UKURAN_BUFFER` | Pola interleaving | Penjelasan |
|---|---|---|
| **1** | Selalu `PCPCPC…` (bergantian ketat) | Buffer hanya muat 1 item: setelah `P` buffer penuh sehingga operasi berikutnya **pasti** `C`, dan setelah `C` buffer kosong sehingga operasi berikutnya **pasti** `P`. Producer paling sering menunggu, karena kedua producer berebut satu slot dan harus menunggu consumer yang lebih lambat. Producer dan consumer berjalan **serentak** ("lockstep"). |
| **3** | Hampir bergantian, sesekali `PPP` atau `CCC` pendek | Ada sedikit ruang penyangga, sehingga producer bisa maju paling banyak 3 item sebelum terhalang. Frekuensi menunggu producer turun dibanding ukuran 1, tetapi masih sering terjadi karena consumer lebih lambat. |
| **10** | Kelompok `P` yang lebih panjang di awal, lalu kelompok `C` panjang di akhir | Buffer cukup besar untuk menampung hampir seluruh produksi, sehingga producer **jarang atau tidak pernah menunggu**. Producer dan consumer menjadi **terlepas-ikat** (decoupled). Isi buffer yang tercapai juga jauh lebih besar (mendekati 8–9 item). Di akhir, ketika producer sudah selesai, consumer menghabiskan sisa isi buffer (`CCCC…`). |



In [9]:
import threading
import time
import random

def eksperimen(ukuran_buffer, n_producer=2, n_consumer=2, item_per_producer=8):
    total_item = n_producer * item_per_producer
    assert total_item % n_consumer == 0, "total item harus habis dibagi jumlah consumer"
    item_per_consumer = total_item // n_consumer

    buffer = []
    mutex = threading.Lock()
    slot_kosong = threading.Semaphore(ukuran_buffer)
    slot_terisi = threading.Semaphore(0)

    log = []   # urutan operasi pada buffer: ("P"/"C", isi buffer setelah operasi)
    stat = {"producer_menunggu": 0, "consumer_menunggu": 0}

    def producer(pid):
        for i in range(item_per_producer):
            time.sleep(random.uniform(0.01, 0.03))        # waktu produksi (cepat)
            if not slot_kosong.acquire(blocking=False):   # buffer penuh -> harus menunggu
                with mutex:
                    stat["producer_menunggu"] += 1
                slot_kosong.acquire()
            with mutex:
                buffer.append(f"P{pid}-item{i}")
                log.append(("P", len(buffer)))
            slot_terisi.release()

    def consumer(cid):
        for _ in range(item_per_consumer):
            if not slot_terisi.acquire(blocking=False):   # buffer kosong -> harus menunggu
                with mutex:
                    stat["consumer_menunggu"] += 1
                slot_terisi.acquire()
            with mutex:
                buffer.pop(0)
                log.append(("C", len(buffer)))
            slot_kosong.release()
            time.sleep(random.uniform(0.03, 0.06))        # waktu konsumsi (lebih lambat)

    threads  = [threading.Thread(target=producer, args=(i + 1,)) for i in range(n_producer)]
    threads += [threading.Thread(target=consumer, args=(i + 1,)) for i in range(n_consumer)]
    for t in threads: t.start()
    for t in threads: t.join()

    pola = "".join(j for j, _ in log)
    isi_maks = max(isi for _, isi in log)
    run = 1 + sum(1 for x, y in zip(pola, pola[1:]) if x != y)   # jumlah "kelompok" P/C berurutan
    return {
        "ukuran": ukuran_buffer, "pola": pola, "isi_maks": isi_maks,
        "p_menunggu": stat["producer_menunggu"], "c_menunggu": stat["consumer_menunggu"],
        "rata_run": len(pola) / run,
        "sisa": len(buffer),
    }

random.seed(7)
hasil_semua = []
for ukuran in (1, 3, 10):
    h = eksperimen(ukuran)
    hasil_semua.append(h)
    print(f"--- UKURAN_BUFFER = {ukuran} (2 producer + 2 consumer, 16 item) ---")
    print("Urutan operasi (P = taruh, C = ambil):")
    print(h["pola"])
    print(f"Sisa buffer di akhir: {h['sisa']} (harus 0)\n")

print("RINGKASAN")
print(f"{'ukuran':>6} | {'isi maks':>8} | {'P menunggu':>10} | {'C menunggu':>10} | {'rata2 panjang run':>17}")
print("-" * 65)
for h in hasil_semua:
    print(f"{h['ukuran']:>6} | {h['isi_maks']:>8} | {h['p_menunggu']:>10} | {h['c_menunggu']:>10} | {h['rata_run']:>17.2f}")

--- UKURAN_BUFFER = 1 (2 producer + 2 consumer, 16 item) ---
Urutan operasi (P = taruh, C = ambil):
PCPCPCPCPCPCPCPCPCPCPCPCPCPCPCPC
Sisa buffer di akhir: 0 (harus 0)

--- UKURAN_BUFFER = 3 (2 producer + 2 consumer, 16 item) ---
Urutan operasi (P = taruh, C = ambil):
PCPCPPPCPCPCPCPCPCPCPCPCPCPCPCCC
Sisa buffer di akhir: 0 (harus 0)

--- UKURAN_BUFFER = 10 (2 producer + 2 consumer, 16 item) ---
Urutan operasi (P = taruh, C = ambil):
PCPCPPPPCPCPPPPCCPPPCPCPCCCCCCCC
Sisa buffer di akhir: 0 (harus 0)

RINGKASAN
ukuran | isi maks | P menunggu | C menunggu | rata2 panjang run
-----------------------------------------------------------------
     1 |        1 |         13 |          2 |              1.00
     3 |        3 |          9 |          2 |              1.14
    10 |        8 |          0 |          2 |              2.00


### Soal 4 — Refleksi Singkat (maks. 150 kata)

Dari ketiga masalah (deadlock, livelock, starvation), menurut kalian mana yang **paling berbahaya** untuk aplikasi produksi nyata (misalnya sistem perbankan atau e-commerce), dan mana yang **paling sulit dideteksi** saat testing? Jelaskan alasannya.

#### Jawaban

Menurut saya, deadlock paling berbahaya untuk sistem perbankan atau e-commerce. Begitu terjadi, thread yang terlibat berhenti permanen sambil tetap memegang lock, sehingga transaksi menggantung, pool koneksi habis, dan dampaknya menjalar ke bagian lain sampai layanan harus di-restart. Livelock juga fatal, tetapi CPU yang terus sibuk membuatnya lebih cepat terlihat di monitoring.

Starvation paling sulit dideteksi saat testing. Tidak ada error, tidak ada hang, semua tes fungsional lulus, dan hasil akhirnya benar. Masalah baru muncul pada beban tinggi sebagai request tertentu yang lambat sekali, dan sulit direproduksi karena bergantung pada penjadwalan thread. Deadlock juga bergantung pada timing, tetapi gejalanya jelas (macet total) sehingga mudah ditangkap dengan watchdog timeout.